# Submit the managed AutoML experiment

[Guide](README.md) · [00: prepare data](00_prepare_data.ipynb) · [01: experiment](01_experiment.ipynb) · [02: results](02_results.ipynb) · [03: predictor](03_predictor.ipynb)

Complete [00 — Prepare data](00_prepare_data.ipynb) first. This notebook submits the installed GA managed pipeline, with the existing Connection Secret for training and test data. It does not compile a custom training pipeline or train AutoGluon locally.

In [1]:
from pathlib import Path
import sys

# Works when Jupyter starts in either the repo root or automl/.
automl_dir = Path.cwd() if Path('workflow.py').exists() else Path.cwd() / 'automl'
sys.path.insert(0, str(automl_dir.resolve()))
import pandas as pd
from IPython.display import display, Markdown, FileLink
import workflow as wf

settings = wf.settings()  # Never display this mapping: it may contain credentials.

## Choose experiment settings and review the inputs

Edit the next cell to compare presets or metrics. Values start from `.env`; changes in this cell apply to this notebook session without rewriting `.env`. `speed` has a 45-minute selection budget, `balanced` 180 minutes; refit, evaluation, and scheduling add time. Prefer `roc_auc` or `f1` for imbalanced fraud data. `top_n` selects 1–10 predictors for refitting/evaluation.

The preview shows **requested** parameters next to any **saved run** parameters. To apply a changed setting or a new data upload, submit a new run explicitly. Resuming a saved run keeps its original configuration.

In [2]:
settings = wf.settings()
EXPERIMENT_NAME = settings.get('EXPERIMENT_NAME', 'fraud-detection-automl')
EXPERIMENT_OPTIONS = {
    'preset': settings.get('AUTOML_PRESET', 'speed'),       # speed | balanced
    'metric': settings.get('AUTOML_EVAL_METRIC', 'roc_auc'),  # roc_auc | f1 | accuracy
    'top_n': settings.get('AUTOML_TOP_N', '1'),             # integer from 1 to 10
}
# Example: EXPERIMENT_OPTIONS['preset'] = 'balanced'
# Connection credentials and infrastructure settings stay in automl/.env.

In [3]:
# Reload the manifest after each new upload in notebook 00.
manifest = wf.read('data_manifest.json')
arguments = wf.pipeline_arguments(manifest, **EXPERIMENT_OPTIONS)
dataset_rows = []
for role in ('train', 'test'):
    dataset = manifest['datasets'][role]
    dataset_rows.append({'role': role, 'S3 URI': dataset['uri'], 'rows': dataset['rows'],
                         'fraud_rate': dataset['fraud_rate']})
display(pd.DataFrame(dataset_rows).style.format({'fraud_rate': '{:.2%}'}))
print('Training includes validation:', manifest['merged_validation'])
print('Recorded Connection Secret:', manifest['connection'])
print('Requested experiment name:', EXPERIMENT_NAME)
preview = pd.DataFrame({'requested': pd.Series(arguments)})
if (wf.OUTPUTS / 'run.json').exists():
    saved_run = wf.read('run.json')
    preview['saved run'] = pd.Series(saved_run['arguments'])
    print('Saved run:', saved_run['run_id'], '| Namespace:', saved_run['project'])
    print('Saved experiment name:', saved_run.get('experiment_name', '(not recorded by this older run)'))
    print('START_NEW_RUN=False reuses that run, including its original inputs and settings.')
display(preview)

,role,S3 URI,rows,fraud_rate
0,train,s3://ai-eng-cracow/fraud-detection/automl/data/20261029T115442Z-a4acdec5/train.csv,800000,8.75%
1,test,s3://ai-eng-cracow/fraud-detection/automl/data/20261029T115442Z-a4acdec5/test.csv,200000,8.69%


Training includes validation: True
Recorded Connection Secret: awss3
Requested experiment name: fraud-detection-automl
Saved run: 7c7e79be-552d-4959-9827-88e564970698 | Namespace: mszewczy
Saved experiment name: (not recorded by this older run)
START_NEW_RUN=False reuses that run, including its original inputs and settings.


,requested,saved run
train_data_secret_name,awss3,awss3
train_data_bucket_name,ai-eng-cracow,ai-eng-cracow
train_data_file_key,fraud-detection/automl/data/20261029T115442Z-a...,fraud-detection/automl/data/20261029T075342Z-8...
test_data_bucket_name,ai-eng-cracow,ai-eng-cracow
test_data_file_key,fraud-detection/automl/data/20261029T115442Z-a...,fraud-detection/automl/data/20261029T075342Z-8...
label_column,fraud,fraud
task_type,binary,binary
positive_class,1,1
eval_metric,roc_auc,roc_auc
preset,speed,speed


## Start or resume

Default: `speed`, ROC-AUC, top one predictor. The pipeline handles preprocessing, model search and refit. Allow time beyond the selection budget for scheduling, refit and evaluation.

The cell resumes a saved run by default. To compare `balanced`, copy `outputs/` to a backup outside the repository, keeping `outputs/data_manifest.json` in place; change `.env`, refresh configuration and explicitly set `START_NEW_RUN=True`. Set it back to `False` afterwards. A failed preflight indicates an incompatible installed pipeline; do not omit `test_data_*` to work around it.

Set `RHOAI_DASHBOARD_URL` to the dashboard base URL to show Run details and AutoML results links. It is separate from the pipeline API endpoint in `KFP_ENDPOINT`.

In [4]:
client = wf.kfp_client()
START_NEW_RUN = False  # True submits another run on EVERY execution; switch back afterwards.
run = wf.submit_experiment(client, manifest, new_run=START_NEW_RUN,
                           experiment_name=EXPERIMENT_NAME, **EXPERIMENT_OPTIONS)
print('Run ID:', run['run_id'])
print('Pipeline version:', run['pipeline_version_id'])
print('Preset:', run['arguments']['preset'])
print('Metric:', run['arguments']['eval_metric'], '| Top N:', run['arguments']['top_n'])
links = wf.run_links(run)
if links:
    for label, url in links.items():
        display(Markdown(f'[{label}]({url})'))
else:
    print('Set RHOAI_DASHBOARD_URL in automl/.env to show Run details and AutoML results links.')

Reusing saved run 7c7e79be-552d-4959-9827-88e564970698. To apply changed inputs or settings, set START_NEW_RUN=True for one execution.
Run ID: 7c7e79be-552d-4959-9827-88e564970698
Pipeline version: fc470922-197f-4365-8652-7d9233ce6c34
Preset: speed
Metric: roc_auc | Top N: 1


[Run details](https://rh-ai.apps.rosa.ai-eng-prod.wrh7.p3.openshiftapps.com/develop-train/pipelines/runs/mszewczy/runs/7c7e79be-552d-4959-9827-88e564970698)

[AutoML results](https://rh-ai.apps.rosa.ai-eng-prod.wrh7.p3.openshiftapps.com/develop-train/automl/results/mszewczy/7c7e79be-552d-4959-9827-88e564970698)

## Monitor the existing run

This cell creates a fresh pipeline client using current configuration, monitors the saved run ID, and saves status. Interrupting it or hitting the local timeout does not cancel training. Run it again to resume monitoring. If the pipeline API returns HTTP 401, renew your OpenShift login if needed, update `RHOAI_TOKEN` in `automl/.env` using the current `oc whoami -t` token, then rerun this cell. Workbench/kernel environment variables override `.env`. Do not submit another run to resume monitoring. Inspect failed task logs and pod termination reasons to distinguish an OOM from other failures.

In [5]:
run = wf.monitor_experiment()  # Fresh credentials; resumes the ID in outputs/run.json.
display(pd.Series({key: run.get(key) for key in
                   ['run_id', 'state', 'submitted_at', 'duration_seconds']}))

12:29:47 7c7e79be-552d-4959-9827-88e564970698: SUCCEEDED


run_id              7c7e79be-552d-4959-9827-88e564970698
state                                          SUCCEEDED
submitted_at            2026-10-29T07:54:19.151165+00:00
duration_seconds                                  2381.0
dtype: object

### Optional: prepare a ticket comment after failure

If monitoring failed, run `print(wf.ticket_report())` in a new cell to capture the run and dataset evidence. It leaves scores and serving status pending. Add observed pod/log evidence yourself.

Next: [02_results.ipynb](02_results.ipynb)